# Step 4. Transfer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/younesStrittmatter/sweetpea-starter/blob/main/tutorial/notebooks/4_transfer.ipynb)

A backup for the tutorial: the same step without an assistant. Run the cells from top to bottom.

In [ ]:
%pip install -q sweetpea pandas

**The question.** Does adapting to frequent conflict make it harder to switch to another task?

**Prediction if shielding and switching are one control setting.** The switch cost is larger in a mostly incongruent context than in a mostly congruent one.

**The alternative to rule out.** A mostly incongruent context is simply harder, so everything is slower there.

**The analysis.** The switch cost (switch against repeat trials) in mostly incongruent mini-blocks against mostly congruent mini-blocks.

In [ ]:
import pandas as pd
from sweetpea import (Factor, DerivedLevel, WithinTrial, Transition, Exclude, MinimumTrials,
                      CrossBlock, synthesize_trials, CMSGen)

colours = ["red", "green", "blue"]

# regular factors: set freely on every trial
colour = Factor("colour", colours)      # the ink
word = Factor("word", colours)          # what the word says
task = Factor("task", ["name the colour", "read the word"])        # new: a second task, with a cue

### The task transition

A transition factor: does the task repeat or switch from the trial before? Without repeat trials in the same context there is no switch cost to measure.

In [ ]:
def task_repeats(task):
    return task[0] == task[-1]


def task_switches(task):
    return task[0] != task[-1]


transition = Factor("transition", [
    DerivedLevel("repeat", Transition(task_repeats, [task])),
    DerivedLevel("switch", Transition(task_switches, [task])),
])

### One mini-block

Congruency (weighted as in step 3), the task and the transition are crossed: 16 combinations, plus a first trial. The ink colour is left out of the crossing: with it a mini-block would be three times as long.

In [ ]:
WEIGHTS = {"mostly congruent": (3, 1), "mostly incongruent": (1, 3)}


def same(colour, word):
    return colour == word


def different(colour, word):
    return colour != word


def mini_block(context):
    congruent_weight, incongruent_weight = WEIGHTS[context]
    congruency = Factor("congruency", [
        DerivedLevel("congruent", WithinTrial(same, [colour, word]), congruent_weight),
        DerivedLevel("incongruent", WithinTrial(different, [colour, word]), incongruent_weight),
    ])
    return CrossBlock(design=[colour, word, task, congruency, transition],
                      crossing=[congruency, task, transition],
                      constraints=[MinimumTrials(17)])

### A session of four mini-blocks

In [ ]:
blocks = []
for number, context in enumerate(["mostly congruent", "mostly incongruent", "mostly incongruent", "mostly congruent"], start=1):
    trials = pd.DataFrame(synthesize_trials(mini_block(context), 1, CMSGen)[0])
    trials.insert(0, "context", context)
    trials.insert(0, "block", number)
    blocks.append(trials)
session = pd.concat(blocks, ignore_index=True)
counted = session[session["transition"] != ""]       # the first trial of a mini-block has no transition
print(len(session), "trials")
print(pd.crosstab(counted["context"], [counted["transition"], counted["congruency"]]))

### What gave way?

In [ ]:
print(pd.crosstab(counted["context"], counted["colour"]))       # not crossed: is the ink colour even?

**Check.** Is there a repeat baseline in each context? How many incongruent switch trials does a mostly congruent mini-block have? Is the ink colour even?

Each question added a factor, and each factor multiplies the design. At some point not everything fits, and the design becomes a choice about what to leave open.